# 成长因子单因子测试 · BigQuant Notebook

复现华泰证券《单因子测试之成长类因子》的完整流程：**12 个成长因子 → 预处理 → IC / 分层 / 回归 / 稳健性**。

- 样本：2020-01 ~ 2025-12，A 股（剔除北交所、ST、停牌、次新）
- 数据：优先读取 `/bq_export_v41/`；不存在时自动改用 `dai` 查询
- 运行：菜单 **Run All** 即可，无需修改代码

| 章节 | 内容 |
|---|---|
| 0 | 参数配置 |
| 1 | 工具函数（与平台单文件脚本同源） |
| 2 | 数据加载与股票池 |
| 3 | 因子构建 |
| 4 | 预处理 |
| 5 | IC 分析 |
| 6 | 分层回测 |
| 7 | 回归法 |
| 8 | IC 衰减 |
| 9 | 分年度 / 分市值 |
| 10 | 稳健性：换手、成本、FDR |
| 11 | 两个关键问题（\|t\| 与收益 t、3Y 反向） |
| 12 | 结果自检 |
| 13 | 结果总览（读取全部产出并可视化） |

## 0. 参数配置

- `DATA_DIR`：数据目录，不存在会自动探测或回退 `dai`
- `USE_DAI`：强制走平台查询
- `SAMPLE_N`：>0 时只取前 N 只股票，用于快速验证
- `OUT_DIR`：图表与表格输出目录

In [ ]:
import os, sys, time, math, warnings, json, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib

# Notebook 内优先用 inline 后端；脚本模式退化为 Agg
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    matplotlib.use("Agg")

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

DATA_DIR = "/bq_export_v41"
USE_DAI = False
START, END = "2020-01-01", "2025-12-31"
SAMPLE_N = 0
OUT_DIR = Path("./out_notebook")
OUT_DIR.mkdir(parents=True, exist_ok=True)

try:
    from IPython.display import display, Image
except Exception:
    display = print
    Image = None
print("参数就绪，输出目录：", OUT_DIR.resolve())

## 1. 工具函数

以下函数与项目里的 `platform/bq_factor_lab.py` 完全同源，包含数据读取、12 因子构建、
去极值 / 行业+市值中性 / 标准化、IC、分层、WLS 回归与 FDR。**无需修改**。

In [ ]:
from __future__ import annotations

import argparse
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

FACTORS = [
    "Sales_G_q", "Sales_G_ttm", "Sales_G_3y",
    "Profit_G_q", "Profit_G_ttm", "Profit_G_3y",
    "OCF_G_q", "OCF_G_ttm", "OCF_G_3y",
    "ROE_G_q", "ROE_G_ttm", "ROE_G_3y",
]

PANEL_COLS = [
    "date", "instrument", "close", "float_market_cap",
    "st_status", "suspended", "list_days", "is_bj", "sw2021_level1_name",
    "operating_revenue_mrq", "operating_revenue_ttm",
    "net_profit_mrq", "net_profit_ttm",
    "net_cffoa_mrq", "net_cffoa_ttm",
    "total_equity_to_parent_shareholders",
]

MIN_LIST_DAYS = 365
MAD_K = 5.0
N_LAYERS = 5


# ---------------------------------------------------------------------------
# 数据读取
# ---------------------------------------------------------------------------

def load_local(data_dir: str, years=None) -> pd.DataFrame:
    files = sorted(Path(data_dir).glob("panel_*.csv"))
    if years is not None:
        files = [f for f in files if int(f.stem.split("_")[1]) in set(years)]
    if not files:
        raise FileNotFoundError(f"{data_dir} 下没有 panel_*.csv")
    cols = [c for c in PANEL_COLS]
    frames = [pd.read_csv(f, usecols=cols, parse_dates=["date"]) for f in files]
    return pd.concat(frames, ignore_index=True)


def resolve_data_dir(p: str) -> str:
    """在常见位置寻找 bq_export_v41，避免路径写错就报错。"""
    cands = [Path(p), Path("/bq_export_v41"),
             Path.home() / "bq_export_v41",
             Path.cwd() / "bq_export_v41",
             Path("/home/aiuser/bq_export_v41")]
    for c in cands:
        try:
            if c.is_dir() and list(c.glob("panel_*.csv")):
                return str(c)
        except OSError:
            continue

    for base in ("/home/aiuser", "/mnt", "/data"):
        b = Path(base)
        if not b.exists():
            continue
        hits = list(b.rglob("panel_2016.csv"))
        if hits:
            return str(hits[0].parent)

    searched = ", ".join(str(c) for c in cands)
    raise FileNotFoundError(
        f"没找到 panel_*.csv。已搜索：{searched}\n"
        "请用 --data-dir 指定实际目录，或改用 --source dai 直接从平台取数。"
    )


def _to_month_end(df: pd.DataFrame) -> pd.DataFrame:
    """把一个年份的日频数据压成月末截面。"""
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    days = pd.Series(df["date"].unique())
    me = days.groupby(days.dt.to_period("M")).max()
    return df[df["date"].isin(set(me.values))]


def load_dai(start: str, end: str) -> pd.DataFrame:
    """从平台取数：按年拉，逐年压成月末，避免一次性载入日频。"""
    import dai

    s, e = pd.Timestamp(start), pd.Timestamp(end)
    frames = []
    for y in range(s.year, e.year + 1):
        a = max(pd.Timestamp(f"{y}-01-01"), s).strftime("%Y-%m-%d")
        b = min(pd.Timestamp(f"{y}-12-31"), e).strftime("%Y-%m-%d")
        df = dai.query(
            f"SELECT {', '.join(PANEL_COLS)} FROM cn_stock_prefactors "
            f"WHERE date >= '{a}' AND date <= '{b}'",
            filters={"date": [a, b]},
        ).df()
        if df is None or len(df) == 0:
            continue
        snap = _to_month_end(df)
        print(f"      {y}: 日频 {len(df):,d} -> 月频 {len(snap):,d}")
        frames.append(snap)
    if not frames:
        return pd.DataFrame(columns=PANEL_COLS)
    return pd.concat(frames, ignore_index=True)


def load_suspension_dai(start: str, end: str) -> pd.DataFrame:
    """从平台取停牌事件（稀疏）。"""
    import dai

    sql = ("SELECT date, instrument FROM cn_stock_prefactors "
           f"WHERE suspended = 1 AND date >= '{start}' AND date <= '{end}'")
    df = dai.query(sql, filters={"date": [start, end]}).df()
    if df is None or len(df) == 0:
        return pd.DataFrame(columns=["date", "instrument"])
    df["date"] = pd.to_datetime(df["date"])
    return df


def load_suspension(data_dir: str) -> pd.DataFrame:
    p = Path(data_dir) / "suspension_events.csv"
    if not p.exists():
        return pd.DataFrame(columns=["date", "instrument"])
    return pd.read_csv(p, parse_dates=["date"])


def filter_pool(df: pd.DataFrame, min_list_days: int = MIN_LIST_DAYS) -> pd.DataFrame:
    mask = (
        df["is_bj"].fillna(0).eq(0)
        & df["st_status"].fillna(0).le(0)
        & df["suspended"].fillna(0).eq(0)
        & (df["list_days"].fillna(0) >= min_list_days)
    )
    return df[mask].copy()


def exclude_next_day_suspension(pool: pd.DataFrame, susp: pd.DataFrame,
                                max_gap_days: int = 5) -> pd.DataFrame:
    """剔除"截面期下一交易日停牌"的股票。"""
    if susp.empty:
        return pool
    by_date = susp.groupby("date")["instrument"].apply(set).to_dict()
    cal = pd.DatetimeIndex(sorted(set(susp["date"]) | set(pool["date"])))
    drop = set()
    for d in pd.DatetimeIndex(sorted(pool["date"].unique())):
        later = cal[cal > d]
        if len(later) == 0:
            continue
        nd = later[0]
        if (nd - d).days > max_gap_days:
            continue
        for inst in by_date.get(nd, ()):
            drop.add((d, inst))
    if not drop:
        return pool
    keys = pd.Series(list(zip(pool["date"], pool["instrument"])), index=pool.index)
    return pool[~keys.isin(drop)]


def to_wide(df: pd.DataFrame) -> dict:
    cols = [c for c in df.columns if c not in ("date", "instrument")]
    return {c: df.pivot(index="date", columns="instrument", values=c).sort_index()
            for c in cols}


# ---------------------------------------------------------------------------
# 因子
# ---------------------------------------------------------------------------

def _growth(cur, base):
    den = base.abs()
    return ((cur - base) / den.where(den > 1e-9)).replace([np.inf, -np.inf], np.nan)


def _cagr(cur, base, years=3):
    ratio = (cur / base.where(base > 0)).where(cur > 0)
    return (ratio.pow(1.0 / years) - 1.0).replace([np.inf, -np.inf], np.nan)


def _roe(profit, equity):
    return (profit / equity.where(equity > 0)).replace([np.inf, -np.inf], np.nan)


def sanitize(w: dict) -> dict:
    """0 填充、负营收统一视为缺失。"""
    out = dict(w)
    for c in ("operating_revenue_mrq", "operating_revenue_ttm"):
        out[c] = w[c].where(w[c] > 0)
    for c in ("net_profit_mrq", "net_profit_ttm", "net_cffoa_mrq", "net_cffoa_ttm"):
        out[c] = w[c].replace(0.0, np.nan)
    eq = "total_equity_to_parent_shareholders"
    out[eq] = w[eq].where(w[eq] > 0)
    return out


def build_factors(w: dict) -> dict:
    w = sanitize(w)
    out = {}
    for prefix, q, ttm in (
        ("Sales", "operating_revenue_mrq", "operating_revenue_ttm"),
        ("Profit", "net_profit_mrq", "net_profit_ttm"),
        ("OCF", "net_cffoa_mrq", "net_cffoa_ttm"),
    ):
        out[f"{prefix}_G_q"] = _growth(w[q], w[q].shift(12))
        out[f"{prefix}_G_ttm"] = _growth(w[ttm], w[ttm].shift(12))
        out[f"{prefix}_G_3y"] = _cagr(w[ttm], w[ttm].shift(36))

    roe_q = _roe(w["net_profit_mrq"], w["total_equity_to_parent_shareholders"])
    roe_t = _roe(w["net_profit_ttm"], w["total_equity_to_parent_shareholders"])
    out["ROE_G_q"] = _growth(roe_q, roe_q.shift(12))
    out["ROE_G_ttm"] = _growth(roe_t, roe_t.shift(12))
    out["ROE_G_3y"] = _cagr(roe_t, roe_t.shift(36))
    return {k: out[k] for k in FACTORS}


# ---------------------------------------------------------------------------
# 预处理与检验
# ---------------------------------------------------------------------------

def mad_winsorize(df, k=MAD_K):
    med = df.median(axis=1)
    mad = df.sub(med, axis=0).abs().median(axis=1)
    return df.clip(lower=med - k * mad, upper=med + k * mad, axis=0)


def zscore(df):
    std = df.std(axis=1).replace(0, np.nan)
    return df.sub(df.mean(axis=1), axis=0).div(std, axis=0)


def residualize(f, ind, logmv):
    out = {}
    for d in f.index:
        s, i, m = f.loc[d], ind.loc[d], logmv.loc[d]
        mask = s.notna() & i.notna() & m.notna()
        if mask.sum() < 50:
            continue
        X = pd.get_dummies(i[mask], drop_first=True).astype(float)
        X.insert(0, "const", 1.0)
        X["logmv"] = m[mask].values
        y = s[mask].to_numpy()
        b, *_ = np.linalg.lstsq(X.to_numpy(), y, rcond=None)
        out[d] = pd.Series(y - X.to_numpy() @ b, index=s[mask].index)
    return pd.DataFrame(out).T.reindex(index=f.index, columns=f.columns)


def prep(f, ind=None, logmv=None):
    x = zscore(mad_winsorize(f))
    if ind is not None and logmv is not None:
        x = residualize(x, ind, logmv)
    return x


def forward_return(close, periods=1):
    return close.pct_change(periods).shift(-periods)


def ic_series(factor, fwd, min_n=30):
    out = {}
    for d in factor.index:
        if d not in fwd.index:
            continue
        f, r = factor.loc[d], fwd.loc[d]
        m = f.notna() & r.notna()
        if m.sum() >= min_n:
            out[d] = f[m].corr(r[m], method="spearman")
    return pd.Series(out).sort_index()


def ic_stats(ic):
    if len(ic) == 0:
        return {}
    m, s = ic.mean(), ic.std()
    return {"n": len(ic), "mean": m, "std": s, "ir": m / s if s else np.nan,
            "t": m / s * np.sqrt(len(ic)) if s else np.nan,
            "positive_ratio": (ic > 0).mean()}


def assign_layers(factor, ind=None, n=N_LAYERS):
    rows = {}
    for d in factor.index:
        f = factor.loc[d]
        grp = ind.loc[d].reindex(f.index) if ind is not None else pd.Series("all", index=f.index)
        m = f.notna()
        if m.sum() < n:
            continue
        pct = f[m].groupby(grp[m].fillna("NA")).rank(pct=True, method="first")
        layer = (n - 1) - np.minimum((pct * n).astype(int), n - 1)
        s = pd.Series(np.nan, index=f.index)
        s.loc[m] = layer.values
        rows[d] = s
    return pd.DataFrame(rows).T.sort_index()


def layer_returns(factor, fwd, ind=None, n=N_LAYERS):
    layers = assign_layers(factor, ind, n)
    out = {}
    for d in layers.index:
        if d not in fwd.index:
            continue
        lay, r = layers.loc[d], fwd.loc[d]
        m = lay.notna() & r.notna()
        if m.sum():
            out[d] = r[m].groupby(lay[m]).mean()
    return pd.DataFrame(out).T.sort_index().reindex(columns=range(n))


def perf_stats(r):
    r = r.dropna()
    if r.empty:
        return {}
    nav = (1 + r).cumprod()
    ann = nav.iloc[-1] ** (12 / len(r)) - 1
    vol = r.std() * np.sqrt(12)
    return {"ann_return": ann, "ann_vol": vol,
            "sharpe": ann / vol if vol else np.nan,
            "max_drawdown": (nav / nav.cummax() - 1).min(),
            "win_rate": (r > 0).mean(), "n": len(r)}


def wls_month(y, f, ind, w):
    m = y.notna() & f.notna() & ind.notna() & w.notna()
    if m.sum() < 50:
        return np.nan, np.nan
    d = pd.DataFrame({"y": y[m], "f": f[m], "ind": ind[m], "w": w[m]})
    d["wy"] = d["y"] * d["w"]
    d["wf"] = d["f"] * d["w"]
    g = d.groupby("ind")[["w", "wy", "wf"]].sum()
    g["ybar"] = g["wy"] / g["w"]
    g["fbar"] = g["wf"] / g["w"]
    d = d.join(g[["ybar", "fbar"]], on="ind")
    yr, fr = d["y"] - d["ybar"], d["f"] - d["fbar"]
    denom = (d["w"] * fr ** 2).sum()
    if denom <= 0:
        return np.nan, np.nan
    beta = (d["w"] * yr * fr).sum() / denom
    e = yr - beta * fr
    k = d["ind"].nunique() + 1
    sigma2 = (d["w"] * e ** 2).sum() / max(len(d) - k, 1)
    se = np.sqrt(sigma2 / denom)
    return beta, beta / se if se > 0 else np.nan


def bh_fdr(p):
    p = np.asarray(p, float)
    m = len(p)
    order = np.argsort(p)
    adj = p[order] * m / np.arange(1, m + 1)
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    out = np.empty(m)
    out[order] = np.clip(adj, 0, 1)
    return out


# ---------------------------------------------------------------------------
# 绘图
# ---------------------------------------------------------------------------

def setup_font(lang="auto"):
    import matplotlib
    import matplotlib.pyplot as plt
    from matplotlib import font_manager

    cjk = ["Noto Sans CJK SC", "WenQuanYi Zen Hei", "SimHei", "Microsoft YaHei"]
    available = {f.name for f in font_manager.fontManager.ttflist}
    hit = [f for f in cjk if f in available]
    if lang == "auto":
        lang = "zh" if hit else "en"
    if lang == "zh" and hit:
        plt.rcParams["font.sans-serif"] = [hit[0]]
        plt.rcParams["axes.unicode_minus"] = False
    return plt, lang

## 2. 数据加载与股票池

优先读本地 CSV；若目录不存在或没有 `panel_*.csv`，自动改用 `dai` 查询（按年取数并压成月末）。

股票池规则：剔除北交所、ST、停牌、上市不足 12 个月，以及**截面期下一交易日停牌**的股票。

In [ ]:
t0 = time.time()
if USE_DAI:
    raw = load_dai(START, END)
    susp = load_suspension_dai(START, END)
    source = "dai"
else:
    try:
        d = resolve_data_dir(DATA_DIR)
        raw = load_local(d)
        susp = load_suspension(d)
        source = f"local:{d}"
    except Exception as e:
        print("本地数据不可用，改用 dai：", e)
        raw = load_dai(START, END)
        susp = load_suspension_dai(START, END)
        source = "dai"

print(f"数据源：{source}")
print(f"原始：{len(raw):,d} 行，{raw['date'].nunique()} 个截面")

pool = filter_pool(raw)
pool = exclude_next_day_suspension(pool, susp)
if SAMPLE_N:
    keep = (pool.groupby("instrument")["close"].count()
            .sort_values(ascending=False).head(SAMPLE_N).index)
    pool = pool[pool["instrument"].isin(keep)]
    print(f"已抽样 {SAMPLE_N} 只")
print(f"过滤后：{len(pool):,d} 行，{pool['instrument'].nunique():,d} 只，耗时 {time.time()-t0:.1f}s")

wide = to_wide(pool)
print(f"宽表字段：{len(wide)} 个")

## 3. 因子构建

12 个因子 = 4 个财务科目（营收 / 净利润 / 经营现金流 / ROE）× 3 种口径（单季同比 / TTM 同比 / 三年复合）。

- 同比：(X_t − X_{t−12}) / |X_{t−12}|
- 三年复合：(X_t / X_{t−36})^(1/3) − 1，要求两端同为正
- 平台用 0 填充的缺失值、负营收、非正净资产，统一视为缺失

In [ ]:
fac_raw = build_factors(wide)
cover = pd.DataFrame({
    "因子": FACTORS,
    "覆盖率": [fac_raw[k].loc[START:END].notna().mean().mean() for k in FACTORS],
})
display(cover.round(3))

## 4. 预处理

每个截面依次：中位数 ±5×MAD 去极值 → 对行业哑变量 + ln(流通市值) 回归取残差 → z-score。

- `fac`：中性化后的因子，用于 IC 与分层
- `fac_reg`：仅去极值 + 标准化的因子，用于回归法（模型中已含行业哑变量）

In [ ]:
close = wide["close"]
dates = [d for d in close.index if pd.Timestamp(START) <= d <= pd.Timestamp(END)]
close, ind = close.loc[dates], wide["sw2021_level1_name"].loc[dates]
mv = wide["float_market_cap"].loc[dates]
logmv = np.log(mv.where(mv > 0))
fwd = forward_return(close)

fac, fac_reg = {}, {}
for k in FACTORS:
    f = fac_raw[k].loc[dates]
    fac[k] = prep(f, ind, logmv)
    fac_reg[k] = prep(f)

plt, LANG = setup_font("auto")
def L(zh, en):
    return zh if LANG == "zh" else en

print(f"评估期 {dates[0]:%Y-%m-%d} ~ {dates[-1]:%Y-%m-%d}，{len(dates)} 个截面，字体={LANG}")

## 5. IC 分析

IC 为因子截面值与下月收益的 Spearman 相关。关注三个数：

- **IC 均值**：方向与强度
- **t 值**：是否显著
- **IC>0 占比**：方向是否稳定

预期：当季（q）因子为正，三年复合（3y）为负。

In [ ]:
ic_all, rows = {}, []
for k, f in fac.items():
    ic = ic_series(f, fwd)
    ic_all[k] = ic
    st = ic_stats(ic)
    rows.append({"因子": k, "IC": st["mean"], "t": st["t"],
                 "IR": st["ir"], "IC>0": st["positive_ratio"]})

ic_tbl = pd.DataFrame(rows).sort_values("IC", ascending=False).reset_index(drop=True)
display(ic_tbl.round(4))

d = ic_tbl.sort_values("IC")
fig, ax = plt.subplots(figsize=(7.4, 3.8))
ax.barh(d["因子"], d["IC"] * 100,
        color=["#1f4e79" if v > 0 else "#c00000" for v in d["IC"]])
ax.axvline(0, color="#333333", lw=0.8)
ax.set_xlabel(L("IC 均值（%）", "Mean IC (%)"))
ax.set_title(L("12 个成长因子的 IC 均值", "Mean IC of 12 growth factors"), loc="left")
fig.tight_layout()
fig.savefig(OUT_DIR / "fig_ic.png", dpi=200, bbox_inches="tight")
plt.show()

print("当季因子全部为正，三年复合因子为负。")

## 6. 分层回测

每月末在行业内按因子值分 5 层，层内等权；多空 = 第 1 层 − 第 5 层。

成本按**真实持仓换手**估算（双边千三），而不是假设 100% 换手。

In [ ]:
layer_rows, navs = [], {}
for k, f in fac.items():
    lay = layer_returns(f, fwd, ind)
    ls = lay[0] - lay[4]

    layers = assign_layers(f, ind)
    hold = {d: set(layers.columns[layers.loc[d] == 0]) for d in layers.index}
    ds = sorted(hold)
    to = [1 - len(hold[a] & hold[b]) / len(hold[a]) for a, b in zip(ds[:-1], ds[1:]) if hold[a]]
    avg_to = float(np.mean(to)) if to else np.nan

    net = ls - avg_to * 0.003
    p_top, p_g, p_n = perf_stats(lay[0]), perf_stats(ls), perf_stats(net)
    m = lay.mean()
    mono = -m.corr(pd.Series(range(5), index=m.index), method="spearman")

    layer_rows.append({"因子": k, "TOP年化": p_top.get("ann_return"),
                       "多空毛": p_g.get("ann_return"), "多空净": p_n.get("ann_return"),
                       "夏普": p_g.get("sharpe"), "回撤": p_g.get("max_drawdown"),
                       "换手": avg_to, "单调性": mono})
    if k.endswith("_q"):
        navs[k] = (1 + ls).cumprod()

layer_tbl = pd.DataFrame(layer_rows)
display(layer_tbl.round(3))

fig, ax = plt.subplots(figsize=(7.4, 3.4))
for k, c in zip(["ROE_G_q", "Profit_G_q", "Sales_G_q", "OCF_G_q"],
                ["#1f4e79", "#c00000", "#7f7f7f", "#c55a11"]):
    ax.plot(navs[k].index, navs[k].values, label=k, color=c, lw=1.2)
ax.axhline(1, color="#999999", lw=0.7, ls="--")
ax.set_title(L("当季因子的多空净值", "Long-short NAV of q factors"), loc="left")
ax.legend(fontsize=7, frameon=False)
fig.tight_layout()
fig.savefig(OUT_DIR / "fig_nav.png", dpi=200, bbox_inches="tight")
plt.show()

print("当季因子单调性为正，三年复合为负。")

## 7. 回归法

每月做一次加权最小二乘：`r_{t+1} = 行业哑变量 + β·f_t + ε`，权重为 √流通市值。

关注 **|t| 均值**（月度截面显著性）与 **收益序列 t**（长期方向是否稳定）的差异。

In [ ]:
reg_rows, beta_store = [], {}
for k in FACTORS:
    betas, ts = {}, {}
    for d in dates:
        b, t = wls_month(fwd.loc[d], fac_reg[k].loc[d], ind.loc[d], np.sqrt(mv.loc[d]))
        betas[d], ts[d] = b, t
    bs, ts_ = pd.Series(betas).dropna(), pd.Series(ts).dropna()
    beta_store[k] = bs
    reg_rows.append({"因子": k, "|t|均值": ts_.abs().mean(),
                     "|t|>2占比": (ts_.abs() > 2).mean(),
                     "β均值": bs.mean(),
                     "收益t": bs.mean() / bs.std() * np.sqrt(len(bs))})

reg_tbl = pd.DataFrame(reg_rows)
display(reg_tbl.round(4))
print("|t| 均值普遍 >2，但收益序列 t 普遍 <2：横截面显著 ≠ 长期有效。")

## 8. IC 衰减

对 1–6 个月的持有期分别计算 IC。若因子是"短期信息优势"，IC 应快速衰减；
若 IC 不降，说明更像持续性定价偏差或风格暴露。

In [ ]:
horizons = {h: forward_return(close, h).loc[dates] for h in range(1, 7)}
dec = {}
for k, f in fac.items():
    dec[k] = {f"t+{h}": ic_series(f, horizons[h]).mean() for h in range(1, 7)}
dec_tbl = pd.DataFrame(dec).T
display(dec_tbl.round(4))

fig, axes = plt.subplots(1, 2, figsize=(7.6, 3.0), sharey=True)
for ax, group, title in (
    (axes[0], ["Sales_G_q", "Profit_G_q", "OCF_G_q", "ROE_G_q"], L("当季（q）", "q")),
    (axes[1], ["Sales_G_3y", "Profit_G_3y", "OCF_G_3y", "ROE_G_3y"], L("三年复合（3y）", "3y")),
):
    for k in group:
        ax.plot(range(1, 7), dec_tbl.loc[k].values * 100, marker="o", ms=3, lw=1.1, label=k)
    ax.axhline(0, color="#999999", lw=0.7)
    ax.set_title(title, fontsize=9)
    ax.set_xlabel(L("持有期（月）", "Horizon (months)"))
    ax.legend(fontsize=6, frameon=False)
axes[0].set_ylabel("IC (%)")
fig.suptitle(L("IC 衰减：当季不衰减，三年复合持续反向", "IC decay"), x=0.02, ha="left", fontsize=10)
fig.tight_layout()
fig.savefig(OUT_DIR / "fig_decay.png", dpi=200, bbox_inches="tight")
plt.show()

## 9. 分年度 / 分市值

- 分年度：看因子在哪一年有效、哪一年失效
- 分市值：按流通市值三分组，组内计算 IC

In [ ]:
ic_df = pd.DataFrame(ic_all)
yearly = ic_df.groupby(ic_df.index.year).mean()
display(yearly.round(4))

fig, ax = plt.subplots(figsize=(7.4, 3.2))
x = np.arange(len(yearly))
w = 0.26
for i, (k, c) in enumerate(zip(["ROE_G_q", "Profit_G_q", "Profit_G_3y"],
                               ["#1f4e79", "#c00000", "#7f7f7f"])):
    ax.bar(x + (i - 1) * w, yearly[k] * 100, w, label=k, color=c)
ax.axhline(0, color="#333333", lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(yearly.index)
ax.set_ylabel("IC (%)")
ax.set_title(L("分年度 IC：2020、2024 强，2021、2023 弱", "Yearly IC"), loc="left")
ax.legend(fontsize=7, frameon=False)
fig.tight_layout()
fig.savefig(OUT_DIR / "fig_yearly.png", dpi=200, bbox_inches="tight")
plt.show()

size_rows = []
for k in ["ROE_G_q", "Profit_G_q", "Sales_G_q", "OCF_G_q"]:
    for g, label in ((0, "小盘"), (1, "中盘"), (2, "大盘")):
        ics = {}
        for d in dates:
            s = mv.loc[d]
            if s.notna().sum() < 100:
                continue
            q = pd.qcut(s.rank(method="first"), 3, labels=[0, 1, 2])
            sel = (q == g).reindex(fac[k].columns).fillna(False).values
            a, b = fac[k].loc[d][sel], fwd.loc[d][sel]
            m = a.notna() & b.notna()
            if m.sum() >= 30:
                ics[d] = a[m].corr(b[m], method="spearman")
        size_rows.append({"因子": k, "市值组": label, "IC": pd.Series(ics).mean()})
size_tbl = pd.DataFrame(size_rows)
display(size_tbl.pivot(index="因子", columns="市值组", values="IC").round(4))
print("当季因子在小盘股中更有效。")

## 10. 稳健性：成本与多重检验

- **真实换手**：月换手 8%–24%，成本拖累 0.3%–0.9%/年
- **FDR**：对 12 个因子的 IC t 值做 Benjamini–Hochberg 校正

In [ ]:
cost_tbl = layer_tbl[["因子", "换手", "多空毛", "多空净", "夏普"]].copy()
cost_tbl["成本拖累"] = cost_tbl["多空毛"] - cost_tbl["多空净"]
display(cost_tbl.round(4))

from math import erf, sqrt as _sqrt
def _p(x):
    return 2 * 0.5 * (1 - erf(abs(x) / _sqrt(2))) if pd.notna(x) else np.nan

fdr = ic_tbl[["因子", "IC", "t"]].copy()
fdr["p"] = fdr["t"].map(_p)
fdr["p_adj"] = bh_fdr(fdr["p"].fillna(1))
fdr["显著"] = fdr["p_adj"] < 0.05
display(fdr.round(4))

print("FDR 后仅 OCF_G_q 显著。")

## 11. 两个关键问题

**Q1**：为什么月度 |t| 很高，但收益序列 t 很低？——β 符号频繁翻转。
**Q2**：三年复合因子为什么反向？——动量暴露 + "低动量高增长"的价值陷阱。

下面的单元分别给出证据。

In [ ]:
# Q1：β 稳定性
stab = []
for k in FACTORS:
    b = beta_store[k]
    stab.append({"因子": k, "|t|均值": reg_tbl.set_index("因子").loc[k, "|t|均值"],
                 "β均值": b.mean(), "β标准差": b.std(),
                 "收益t": b.mean() / b.std() * np.sqrt(len(b)),
                 "正收益月份": (b > 0).mean(),
                 "符号翻转率": (np.sign(b).diff().abs() > 0).sum() / (len(b) - 1)})
stab_tbl = pd.DataFrame(stab)
display(stab_tbl.round(3))

fig, ax = plt.subplots(figsize=(7.4, 3.4))
d = stab_tbl.sort_values("|t|均值")
y = np.arange(len(d))
ax.barh(y - 0.19, d["|t|均值"], height=0.36, color="#1f4e79", label=L("月度 |t| 均值", "mean |t|"))
ax.barh(y + 0.19, d["收益t"], height=0.36, color="#c00000", label=L("收益序列 t", "return t"))
ax.axvline(2, color="#7f7f7f", lw=0.8, ls="--")
ax.set_yticks(y); ax.set_yticklabels(d["因子"], fontsize=7)
ax.set_title(L("“每月都能排序”不等于“长期能赚钱”", "|t| vs return t"), loc="left")
ax.legend(fontsize=7, frameon=False)
fig.tight_layout()
fig.savefig(OUT_DIR / "fig_tstat.png", dpi=200, bbox_inches="tight")
plt.show()

# Q2：3y 因子的动量暴露与分组
past12 = close.pct_change(12)
fwd12 = close.pct_change(12).shift(-12)
rows = []
for k in ["Sales_G_3y", "Profit_G_3y", "OCF_G_3y", "ROE_G_3y"]:
    cors, qs = {}, {}
    for d in dates:
        a, b = fac_raw[k].loc[d], past12.loc[d]
        m = a.notna() & b.notna()
        if m.sum() >= 50:
            cors[d] = a[m].corr(b[m], method="spearman")
        a2, b2 = fac_raw[k].loc[d], fwd12.loc[d]
        m2 = a2.notna() & b2.notna()
        if m2.sum() >= 100:
            qq = pd.qcut(a2[m2].rank(method="first"), 5, labels=False)
            qs[d] = b2[m2].groupby(qq).mean()
    qdf = pd.DataFrame(qs).T
    rows.append({"因子": k, "动量相关": pd.Series(cors).mean(),
                 **{f"Q{i+1}": qdf[i].mean() for i in range(5)}})
three_tbl = pd.DataFrame(rows)
display(three_tbl.round(3))

fig, ax = plt.subplots(figsize=(7.4, 3.1))
for k, c in zip(["Sales_G_3y", "Profit_G_3y", "ROE_G_3y"], ["#1f4e79", "#c00000", "#7f7f7f"]):
    ax.plot(range(1, 6), three_tbl.set_index("因子").loc[k, [f"Q{i}" for i in range(1, 6)]] * 100,
            marker="o", ms=4, lw=1.2, label=k, color=c)
ax.set_xticks(range(1, 6))
ax.set_xlabel(L("三年增长分组（1=最低，5=最高）", "3y growth quintile"))
ax.set_ylabel(L("未来 12 个月收益（%）", "Forward 12m return (%)"))
ax.set_title(L("高三年增长组的事后收益更低", "Higher 3y growth earns less"), loc="left")
ax.legend(fontsize=7, frameon=False)
fig.tight_layout()
fig.savefig(OUT_DIR / "fig_3y.png", dpi=200, bbox_inches="tight")
plt.show()

## 12. 结果自检

把当前结果与本地全样本基准值比对。**全部 PASS** 才说明平台与本地一致。

In [ ]:
EXPECTED_IC = {
    "ROE_G_q": 0.018423, "Profit_G_q": 0.015466, "Sales_G_q": 0.010876,
    "OCF_G_q": 0.009671, "ROE_G_ttm": 0.005593, "OCF_G_ttm": 0.004107,
    "OCF_G_3y": 0.002594, "Profit_G_ttm": 0.002028, "Sales_G_ttm": -0.001412,
    "ROE_G_3y": -0.002850, "Sales_G_3y": -0.002959, "Profit_G_3y": -0.007921,
}
got = ic_tbl.set_index("因子")["IC"]
bad = []
for k, v in EXPECTED_IC.items():
    diff = abs(float(got[k]) - v)
    flag = "PASS" if diff < 1e-4 else "FAIL"
    if flag == "FAIL":
        bad.append(k)
    print(f"{k:12s} 平台={got[k]:+.6f}  本地={v:+.6f}  diff={diff:.2e}  {flag}")
print("\n结果：", "全部一致 ✅" if not bad else f"存在差异：{bad}")

## 13. 结果总览

读取 `out_notebook/` 下的全部产出（CSV + PNG），在 Notebook 中统一展示，方便截图或汇报。

In [ ]:
print("产出文件：")
for p in sorted(OUT_DIR.iterdir()):
    print(" -", p.name, f"{p.stat().st_size/1024:.1f} KB")

for name in ["ic_tbl", "layer_tbl", "reg_tbl", "dec_tbl", "fdr"]:
    if name in globals():
        print(f"\n### {name}")
        display(globals()[name].round(4))

print("\n### 图表")
for p in sorted(OUT_DIR.glob("*.png")):
    print(p.name)
    if Image is not None:
        display(Image(filename=str(p)))

## 14. 导出与结论

- 表格导出为 CSV，图表为 PNG，统一打包 `out_notebook.zip`
- 结论：当季因子仍有区分度（IC 1.0%–1.8%、单调性 0.9–1.0），但强度较原文衰减；
  三年复合因子系统性反向；扣真实换手成本后，当季因子多空仍有 2.6%–5.3%。

In [ ]:
tables = {"ic_summary": ic_tbl, "layer_summary": layer_tbl,
          "regression_summary": reg_tbl, "ic_decay": dec_tbl, "fdr_summary": fdr}
for name, df in tables.items():
    df.to_csv(OUT_DIR / f"{name}.csv", index=False, encoding="utf-8-sig")

with zipfile.ZipFile(OUT_DIR / "out_notebook.zip", "w") as z:
    for p in OUT_DIR.iterdir():
        if p.suffix in (".csv", ".png"):
            z.write(p, p.name)

print("已导出到", OUT_DIR.resolve())
print("总耗时：", f"{time.time()-t0:.1f}s")